# 📊 Visualization Notebook
## News Clickbait Detection + Summarization — Deep Learning Mini Project
This notebook produces all visualizations required by the rubric:
- Training loss & validation loss curves
- Accuracy over epochs
- Confusion matrix
- Classification report heatmap
- Probability distribution of predictions
- ROUGE score bar chart (summarization)

In [ ]:
import os
import re
import string
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.nn as nn
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, roc_curve, auc
)
from torch.utils.data import DataLoader, TensorDataset

# ── import model class (file must be in same directory or on path) ──
import sys
sys.path.insert(0, '..')   # adjust if needed
from clickbait_lstm import ClickbaitLSTM

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ── consistent style ──
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
SAVE_DIR = 'plots'
os.makedirs(SAVE_DIR, exist_ok=True)

---
## 1. Data Loading & Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df1 = pd.read_csv('Data/Clickbait/train1.csv')
df2 = pd.read_csv('Data/Clickbait/train2.csv')

df1 = df1.rename(columns={'headline': 'title', 'clickbait': 'label'})
df2['label'] = df2['label'].astype(str).str.strip().str.lower().replace({'clickbait': 1, 'news': 0})

df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=['title', 'label'])
df['label'] = df['label'].astype(int)
df['title'] = df['title'].astype(str).apply(clean_text)
df = df[df['title'].str.strip() != '']

print(f'Total samples: {len(df)}')
print(df['label'].value_counts())

---
## 2. Class Distribution

In [ ]:
counts = df['label'].value_counts().rename({0: 'Not Clickbait', 1: 'Clickbait'})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# bar chart
bars = axes[0].bar(counts.index, counts.values, color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Class Distribution (Bar)', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, counts.max() * 1.15)

# pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Distribution (Pie)', fontweight='bold')

plt.suptitle('Dataset Class Balance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Train Model & Record History

In [ ]:
# ── split ──
X_train, X_temp, y_train, y_temp = train_test_split(
    df['title'], df['label'], test_size=0.3, random_state=42, stratify=df['label'])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# ── vocab ──
counter = Counter(' '.join(X_train).split())
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, freq in counter.items():
    if freq >= 2:
        vocab[word] = len(vocab)

def encode(text, max_len=20):
    tokens = text.split()[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    return ids + [0] * (max_len - len(ids))

X_train_enc = torch.tensor([encode(t) for t in X_train])
X_val_enc   = torch.tensor([encode(t) for t in X_val])
X_test_enc  = torch.tensor([encode(t) for t in X_test])
y_train_t   = torch.tensor(y_train.values, dtype=torch.float)
y_val_t     = torch.tensor(y_val.values,   dtype=torch.float)
y_test_t    = torch.tensor(y_test.values,  dtype=torch.float)

train_loader = DataLoader(TensorDataset(X_train_enc, y_train_t), batch_size=32, shuffle=True)

# ── model ──
model = ClickbaitLSTM(len(vocab), 64, 128).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
pos_weight = torch.tensor([len(y_train_t[y_train_t == 0]) / max(len(y_train_t[y_train_t == 1]), 1)]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── training loop with history recording ──
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
patience, counter_es = 3, 0

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_out  = model(X_val_enc.to(DEVICE))
        val_loss = criterion(val_out, y_val_t.to(DEVICE)).item()
        val_preds = (torch.sigmoid(val_out).cpu().numpy() >= 0.5).astype(int)
        val_acc   = accuracy_score(y_val.values, val_preds)

    avg_train = total_loss / len(train_loader)
    history['train_loss'].append(avg_train)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1:2d} | Train Loss: {avg_train:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_loss < best_val_loss - 0.001:
        best_val_loss = val_loss
        counter_es = 0
        torch.save(model.state_dict(), 'best_model_viz.pt')
    else:
        counter_es += 1
        if counter_es >= patience:
            print('Early stopping triggered')
            break

model.load_state_dict(torch.load('best_model_viz.pt'))
print('\nTraining complete.')

---
## 4. Training & Validation Loss Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss
axes[0].plot(epochs_ran, history['train_loss'], 'o-', color='#4C72B0', label='Train Loss', linewidth=2)
axes[0].plot(epochs_ran, history['val_loss'],   's--', color='#DD8452', label='Val Loss',   linewidth=2)
axes[0].set_title('Loss Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[0].legend()

# Accuracy
axes[1].plot(epochs_ran, history['val_acc'], 'D-', color='#55A868', label='Val Accuracy', linewidth=2)
axes[1].set_title('Validation Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].legend()

plt.suptitle('LSTM Clickbait Detector — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/02_loss_accuracy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Confusion Matrix

In [ ]:
model.eval()
with torch.no_grad():
    test_out = model(X_test_enc.to(DEVICE))
    probs = torch.sigmoid(test_out).cpu().numpy().flatten()
    preds = (probs >= 0.5).astype(int)

y_true = y_test_t.numpy()

cm = confusion_matrix(y_true, preds)
labels = ['Not Clickbait', 'Clickbait']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels,
    linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.75}, ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTest Accuracy:', accuracy_score(y_true, preds))
print('\nClassification Report:')
print(classification_report(y_true, preds, target_names=labels))

---
## 6. Classification Report Heatmap

In [ ]:
report = classification_report(y_true, preds, target_names=labels, output_dict=True)
report_df = pd.DataFrame(report).T.drop(columns='support', errors='ignore')
report_df = report_df.loc[labels]   # only per-class rows

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(
    report_df.astype(float), annot=True, fmt='.3f',
    cmap='YlGnBu', vmin=0, vmax=1,
    linewidths=0.5, linecolor='white', ax=ax
)
ax.set_title('Per-Class Metrics (Precision / Recall / F1)', fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_classification_report_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, probs)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#4C72B0', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='grey', lw=1.5, linestyle='--', label='Random classifier')
ax.fill_between(fpr, tpr, alpha=0.08, color='#4C72B0')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Clickbait Detector', fontweight='bold', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/05_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Prediction Probability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(probs[y_true == 0], bins=40, alpha=0.65, color='#4C72B0',
        label='Not Clickbait (true)', edgecolor='white')
ax.hist(probs[y_true == 1], bins=40, alpha=0.65, color='#DD8452',
        label='Clickbait (true)',     edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
ax.set_xlabel('Predicted Clickbait Probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Prediction Probability Distribution', fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06_prob_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. ROUGE Scores — T5 Summarization
> Run `summarization_pretrained.py` first to generate `results/rouge_scores.txt`, or paste your scores below.

In [ ]:
# ── Try to load from saved file, else use manually entered values ──
try:
    import ast
    with open('results/rouge_scores.txt', 'r') as f:
        rouge_results = ast.literal_eval(f.read())
    print('Loaded ROUGE scores from file.')
except FileNotFoundError:
    # ← paste your actual scores here if file not present
    rouge_results = {'rouge1': 0.31, 'rouge2': 0.12, 'rougeL': 0.27}
    print('Using manually entered ROUGE scores (file not found).')

rouge_names  = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
rouge_values = [rouge_results['rouge1'], rouge_results['rouge2'], rouge_results['rougeL']]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4C72B0', '#55A868', '#DD8452']
bars = ax.bar(rouge_names, rouge_values, color=colors, edgecolor='white', width=0.45)
for bar, val in zip(bars, rouge_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, max(rouge_values) * 1.2)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('T5-Small ROUGE Scores on Test Set', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/07_rouge_scores.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Summary of All Saved Plots

In [ ]:
saved = sorted(os.listdir(SAVE_DIR))
print(f'Saved {len(saved)} plots to ./{SAVE_DIR}/')
for f in saved:
    print(' •', f)